# XAI SME Threat Detection - Full Pipeline

This notebook walks through the entire study end to end, in the exact order it was run. Each code cell is preceded by a short brief explaining **what** it does and **why**. Run cells top to bottom.

Pipeline: **XGBoost (detect) -> SHAP (explain) -> Claude (translate)**, evaluated on **IDS2025** and **UNSW-NB15**, then stress-tested for adversarial robustness (RQ6).

> Paths assume this notebook runs from `notebooks/`, so data and models are referenced with `../`.

## 1. Load and inspect IDS2025
**What:** loads the dataset and prints its shape, columns, and class balance.  
**Why:** confirms the file loaded correctly and shows the label column is `newLabel` (not `Label`) and that the classes are rebalanced (Normal ~28.5%, versus ~83% benign in raw CICIDS2017).

In [ ]:
import pandas as pd

df = pd.read_excel('../data/IDS2025.xlsx')
print(df.shape)              # (91830, 80)
print(df.columns.tolist())
print(df['newLabel'].value_counts())

## 2. Data-quality checks
**What:** counts nulls and infinities, and checks dtypes.  
**Why:** CICIDS-family data often has infinity values in flow-rate columns. Here they are negligible (77 nulls, 177 infinities), safe to drop rather than impute.

In [ ]:
print(df.isnull().sum().sum())                               # 77
print(df.isin([float('inf'), float('-inf')]).sum().sum())    # 177
print(df.dtypes.value_counts())                              # 55 int, 24 float, 1 object

## 3. Clean nulls and infinities
**What:** replaces infinities with NaN, then drops affected rows.  
**Why:** removes the small amount of unusable data so the model trains on clean numeric features.

In [ ]:
import numpy as np

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()
print(df.shape)

## 4. Encode the label
**What:** turns the text class names into integers 0-6.  
**Why:** XGBoost needs a numeric target. The printed mapping records which number is which class.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['newLabel'])
print(dict(zip(le.classes_, le.transform(le.classes_))))

## 5. Stratified train/test split
**What:** 80/20 split preserving class proportions, with `random_state=42`.  
**Why:** stratify keeps every class (including tiny Infiltration) present in both sets; the fixed seed makes the split reproducible (73,362 train / 18,341 test).

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['newLabel', 'label_encoded'])
y = df['label_encoded']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)

## 6. Train XGBoost and evaluate (RQ1 / RQ4)
**What:** trains the primary detector and prints the per-class report.  
**Why:** establishes detection performance. Result: ~99.58% accuracy, macro-F1 0.9688. Infiltration recall is low only because it has 6 test samples.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

model = XGBClassifier(objective='multi:softmax', num_class=7,
                      eval_metric='mlogloss', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 7. Baseline comparison (RQ1)
**What:** trains Random Forest and Logistic Regression and compares accuracy + macro-F1.  
**Why:** shows XGBoost is justified. LR gets scaled features (it is scale-sensitive); the trees do not (they split on thresholds). Result: XGB 0.9958 / RF 0.9953 / LR 0.9421 accuracy.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train); rf_pred = rf.predict(X_test)

scaler = StandardScaler()
Xtr_s = scaler.fit_transform(X_train); Xte_s = scaler.transform(X_test)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(Xtr_s, y_train); lr_pred = lr.predict(Xte_s)

for name, pred in [('XGBoost', y_pred), ('Random Forest', rf_pred), ('Logistic Reg', lr_pred)]:
    print(f'{name:15} accuracy={accuracy_score(y_test, pred):.4f}  '
          f'macro-F1={f1_score(y_test, pred, average="macro"):.4f}')

## 8. Leakage checks: duplicates, feature importance, Destination Port ablation
**What:** three independent checks that the high accuracy is genuine, not leakage.  
**Why:** near-perfect accuracy on CICIDS-family data invites a leakage objection. Result: 86 duplicates (0.09%); no single dominant feature; removing Destination Port changes accuracy by <0.03%.

In [ ]:
print('Duplicates:', df.duplicated().sum())

importances = pd.Series(model.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False).head(10))

Xtr2 = X_train.drop(columns=['Destination Port'])
Xte2 = X_test.drop(columns=['Destination Port'])
m2 = XGBClassifier(objective='multi:softmax', num_class=7,
                   eval_metric='mlogloss', random_state=42)
m2.fit(Xtr2, y_train); p2 = m2.predict(Xte2)
print('WITHOUT Destination Port: accuracy=%.4f  macro-F1=%.4f' % (
    accuracy_score(y_test, p2), f1_score(y_test, p2, average='macro')))

## 9. RQ3: false-alarm rate
**What:** counts how many benign Normal flows get flagged as an attack.  
**Why:** false alarms waste a lean SME team's time. Result: 64 of 5,233 = 1.22%.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
ni = list(le.classes_).index('Normal')
fa = cm[ni].sum() - cm[ni, ni]
print(f'Normal flagged as attack: {fa} of {cm[ni].sum()} ({100*fa/cm[ni].sum():.2f}%)')

## 10. Save model + dashboard artifacts
**What:** persists the model and the files the Streamlit dashboard reads.  
**Why:** the dashboard runs as its own process and cannot see notebook memory, so it loads these from disk.

In [ ]:
import joblib, json

joblib.dump(model, '../models/xgboost_ids2025.pkl')
X_test.iloc[:50].to_csv('../data/X_test_sample.csv', index=False)
with open('../models/class_names.json', 'w') as f:
    json.dump(le.classes_.tolist(), f)
print('Saved.')

## 11. SHAP explainer + top-5 features for one alert
**What:** builds a TreeExplainer and extracts the features that drove one prediction.  
**Why:** this is the explainer layer. `shap_values` has shape (100, 79, 7) = samples x features x classes; we slice `[sample, :, predicted_class]` to explain the class the model actually chose.

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test.iloc[:100])
print('SHAP shape:', shap_values.shape)

i = 0
pc = model.predict(X_test.iloc[[i]])[0]
print('Predicted class:', le.classes_[pc])
sample_shap = shap_values[i, :, pc]
top_idx = np.argsort(np.abs(sample_shap))[::-1][:5]
for j in top_idx:
    print(f'{X_test.columns[j]}: {sample_shap[j]:.4f}')

## 12. Claude translation layer
**What:** turns the SHAP top-5 into a plain-language, three-sentence analyst summary.  
**Why:** a non-expert needs an action, not a feature chart. The prompt is constrained (3 sentences, no markdown, max_tokens=200). Requires `ANTHROPIC_API_KEY` in `../.env` as `NAME=value`.

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv('../.env', override=True)
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

def explain_alert(idx, X_test, model, shap_vals, class_names):
    pc = model.predict(X_test.iloc[[idx]])[0]
    s = shap_vals[idx, :, pc]
    tix = np.argsort(np.abs(s))[::-1][:5]
    feats = '\n'.join([f'- {X_test.columns[k]}: {s[k]:.4f}' for k in tix])
    prompt = f'''You are a SOC analyst assistant. Given this network traffic alert
classification and its top contributing features, write a SHORT plain-text
explanation for a dashboard card. No markdown, no headers, no bullet points.

Write exactly 3 sentences:
1. Why this alert was classified this way (plain English, reference the top feature)
2. One specific, concrete action the analyst should take right now
3. One sentence of risk context for a small/medium organization

Classification: {class_names[pc]}
Top contributing features (SHAP values, positive = pushed toward this class):
{feats}
'''
    r = client.messages.create(model='claude-sonnet-4-5', max_tokens=200,
                               messages=[{'role': 'user', 'content': prompt}])
    return class_names[pc], r.content[0].text

cls, text = explain_alert(0, X_test, model, shap_values, le.classes_)
print(f'--- {cls} ---'); print(text)

## 13. RQ2: explanation consistency
**What:** measures how stable the top-5 explanation is across alerts of the same class.  
**Why:** tests whether explanations are trustworthy per class. Result: PortScan 0.772 (structured, consistent) vs Normal 0.298 (heterogeneous).

In [ ]:
from itertools import combinations

by_class = {}
for i in range(100):
    pc = model.predict(X_test.iloc[[i]])[0]
    s = shap_values[i, :, pc]
    by_class.setdefault(pc, []).append(set(np.argsort(np.abs(s))[::-1][:5].tolist()))

for pc, sets in sorted(by_class.items()):
    if len(sets) < 2:
        print(f'  {le.classes_[pc]:15} only {len(sets)} sample'); continue
    jacs = [len(a & b)/len(a | b) for a, b in combinations(sets, 2)]
    print(f'  {le.classes_[pc]:15} mean Jaccard={np.mean(jacs):.3f}  (n={len(sets)})')

## 14. RQ6: adversarial robustness of explanations
**What:** runs the prediction-preserving attack (single-alert demo, then the 100-alert budget sweep).  
**Why:** the central safety finding. It tests whether the explanation can be changed while the verdict stays fixed. Result: 83-88% of explanations disrupted even at the smallest budget. Uses `src/attack_explanations.py`.

In [ ]:
import sys
sys.path.insert(0, '../src')
from attack_explanations import feature_std_vector, show_attack_example, run_budget_sweep

stds = feature_std_vector(X_train)

# Single-alert demo: verdict holds, explanation shifts
show_attack_example(model, explainer, X_test.iloc[[0]], stds,
                    list(X_test.columns), le.classes_, budget=0.5, n_trials=500, seed=1)

# Budget sweep: the RQ6 result table
results = run_budget_sweep(
    model, explainer, X_test.iloc[:100], stds, list(X_test.columns),
    budgets=(0.05, 0.1, 0.25, 0.5, 1.0), n_trials=200,
    save_path='../results/robustness_sweep_100.json')